In [0]:
from pyspark.sql.functions import current_timestamp, lit

ruta_volumen = "/Volumes/agro_catalog/bronce/dp750/"

def ingestar_a_bronze(nombre_archivo, nombre_tabla_destino, descripcion_tabla):
    print(f"Leyendo {nombre_archivo} desde el volumen...")
    
    #lectura en crudo del archivo ams ingreso de los metadatos
    df_crudo = (
        spark.read.json(f"{ruta_volumen}{nombre_archivo}")
        .withColumn("fecha_ingesta", current_timestamp())
        .withColumn("archivo_origen", lit(nombre_archivo))
    )
    
    tabla_delta = f"agro_catalog.bronce.{nombre_tabla_destino}_bronze"
    
    # creacion fromato delta
    (df_crudo.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabla_delta))
    
    spark.sql(f"COMMENT ON TABLE {tabla_delta} IS '{descripcion_tabla}'")
        
    total_registros = df_crudo.count()
    print(f"Tabla creada: {tabla_delta} con {total_registros} registros.\n")

# ingesta 
ingestar_a_bronze(
    "sensor_readings.jsonl", 
    "sensor_readings", 
    "Capa Bronze: Lecturas crudas de sensores de suelo IoT en formato JSON anidado."
)
ingestar_a_bronze(
    "weather_readings.jsonl", 
    "weather_readings", 
    "Capa Bronze: Lecturas crudas de estaciones meteorologicas en formato JSON anidado."
)
ingestar_a_bronze(
    "irrigation_events.jsonl", 
    "irrigation_events", 
    "Capa Bronze: Registro de eventos de riego y apertura de valvulas en bruto."
)
ingestar_a_bronze(
    "alerts.jsonl", 
    "alerts", 
    "Capa Bronze: Alertas operativas crudas generadas por los dispositivos IoT."
)

Leyendo sensor_readings.jsonl desde el volumen...
Tabla creada: agro_catalog.bronce.sensor_readings_bronze con 6190 registros.

Leyendo weather_readings.jsonl desde el volumen...
Tabla creada: agro_catalog.bronce.weather_readings_bronze con 619 registros.

Leyendo irrigation_events.jsonl desde el volumen...
Tabla creada: agro_catalog.bronce.irrigation_events_bronze con 3095 registros.

Leyendo alerts.jsonl desde el volumen...
Tabla creada: agro_catalog.bronce.alerts_bronze con 98 registros.

